**M3 - Modelagem, Avaliação e Análise de Erros - Gustavo Lobato Campos e Rafael Vinicius Tayette da Nobrega**

**Fonte dos dados:** [Kaggle - Solar Power Generation Data (anikannal)](https://www.kaggle.com/datasets/anikannal/solar-power-generation-data)

Data de acesso: 10/07/2026

Este notebook parte da base combinada das Plantas 1 e 2 e implementa a etapa de modelagem preditiva.

Estrutura do notebook:
0. Preparação: base consolidada, engenharia de atributos e split temporal
1. Baseline (piso de desempenho)
2. Experimentos (>= 3 abordagens, CV temporal, média ± desvio)
3. Modelo final (escolha, hiperparâmetros, avaliação no teste)
4. Análise de erros
5. Limitações


**A. Configuração inicial:** 

In [15]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              HistGradientBoostingRegressor, HistGradientBoostingClassifier)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_validate, RandomizedSearchCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, RocCurveDisplay, f1_score)

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titleweight'] = 'bold'
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

RANDOM_STATE = 42
TARGET       = 'AC_POWER'          # variável de origem (kW)
ALVO         = 'AC_POWER_H1'       # alvo deslocado: AC_POWER 1h à frente
PASSO_MIN    = 15                  # granularidade da série (minutos)
HORIZONTE_H  = 1                   # horizonte de previsão (horas)
DATA_CORTE   = pd.Timestamp('2020-06-10')   # fronteira treino/teste (split temporal)
LIMIAR_BAIXA = 0.50                # evento de baixa geração: < 50% da mediana do horário
N_SPLITS     = 5

def rmse(y, yhat):
    return float(np.sqrt(mean_squared_error(y, yhat)))


In [16]:
# Carregamento e merge das duas plantas (idêntico ao Módulo 2)
gen1 = pd.read_csv('dataset/Plant_1_Generation_Data.csv')
wth1 = pd.read_csv('dataset/Plant_1_Weather_Sensor_Data.csv')
gen2 = pd.read_csv('dataset/Plant_2_Generation_Data.csv')
wth2 = pd.read_csv('dataset/Plant_2_Weather_Sensor_Data.csv')

gen1['DATE_TIME'] = pd.to_datetime(gen1['DATE_TIME'], format='%d-%m-%Y %H:%M')
wth1['DATE_TIME'] = pd.to_datetime(wth1['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
gen2['DATE_TIME'] = pd.to_datetime(gen2['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
wth2['DATE_TIME'] = pd.to_datetime(wth2['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')

COLS_CLIMA = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']

def combinar(gen, wth, planta):
    d = pd.merge(gen, wth.drop(columns=['SOURCE_KEY']), on=['DATE_TIME', 'PLANT_ID'], how='left')
    d = d.sort_values('DATE_TIME')
    d[COLS_CLIMA] = d[COLS_CLIMA].interpolate(method='linear', limit_direction='both')
    d['PLANTA'] = planta
    return d

df = pd.concat([combinar(gen1, wth1, 1), combinar(gen2, wth2, 2)], ignore_index=True)
df = df.sort_values(['PLANTA', 'SOURCE_KEY', 'DATE_TIME']).reset_index(drop=True)

print(f"Base consolidada: {df.shape[0]:,} registros x {df.shape[1]} colunas".replace(',', '.'))
print(f"\nPeríodo: {df['DATE_TIME'].min()} a {df['DATE_TIME'].max()}\n")
print(df.groupby('PLANTA').agg(registros=('AC_POWER', 'size'),
                               inversores=('SOURCE_KEY', 'nunique'),
                               ac_max=('AC_POWER', 'max'),
                               dc_max=('DC_POWER', 'max')))


Base consolidada: 136.476 registros x 11 colunas

Período: 2020-05-15 00:00:00 a 2020-06-17 23:45:00

        registros  inversores   ac_max        dc_max
PLANTA                                              
1           68778          22  1410.95  14471.125000
2           67698          22  1385.42   1420.933333


B Engenharia de atributos

Três decisões impactam a matriz de atributos, porque determinam a validade de tudo que vem na sequência.

(a) Vazamento temporal. `AC_POWER` e `DC_POWER` no mesmo instante têm correlação próxima de 1 (Módulo 2, seção 4.4). Um modelo que prevê `AC_POWER(t)` a partir de `DC_POWER(t)` atinge R² ≈ 0,999 e não prevê nada: apenas reaprende a eficiência de conversão do inversor. Por isso o alvo é deslocado para **t + 1h** e todos os atributos são medidos em **t**, ou seja, em um instante anterior. Assim, `DC_POWER(t)` passa a ser um atributo legítimo.

(b) Deslocamentos por junção, não por `shift()`. A série tem lacunas (timestamps ausentes). `groupby().shift(4)` deslocaria 4 *linhas*, não 4 *passos de 15 min*, corrompendo silenciosamente os lags nas bordas das lacunas. Os deslocamentos são feitos por `merge` sobre `DATE_TIME + Δ`, o que é exato na presença de lacunas.

(c) Escala de `DC_POWER` entre plantas. Na Planta 1 `DC_POWER` é reportada uma escala cerca de 10x maior que na Planta 2 (ver `dc_max` acima), consequência de arranjos distintos de string/inversor. Usar a variável bruta com as duas plantas juntas cria um atributo cujo significado muda conforme a planta. Assim, é incluída a versão normalizada pelo máximo de treino da própria planta.

`DAILY_YIELD` e `TOTAL_YIELD` são descartados: são acumuladores monotônicos que codificam a posição no dia e no período de coleta, gerando um atalho que não sobrevive fora da janela de 34 dias.

In [17]:
CHAVE = ['PLANTA', 'SOURCE_KEY']

def deslocar(base, colunas, passos, sufixo):
    """Desloca `colunas` em `passos` * 15 min via merge exato sobre DATE_TIME.
    passos > 0 -> valor passado (lag). passos < 0 -> valor futuro (alvo)."""
    delta = pd.Timedelta(minutes=PASSO_MIN * passos)
    aux = base[CHAVE + ['DATE_TIME'] + colunas].copy()
    aux['DATE_TIME'] = aux['DATE_TIME'] + delta
    aux = aux.rename(columns={c: f'{c}{sufixo}' for c in colunas})
    return base.merge(aux, on=CHAVE + ['DATE_TIME'], how='left')

# --- alvo: AC_POWER 1h à frente (4 passos de 15 min) ---
df = deslocar(df, [TARGET], -4, '_H1')

# --- lags de potência (15, 30, 60 min atrás) ---
for p in [1, 2, 4]:
    df = deslocar(df, ['AC_POWER', 'DC_POWER'], p, f'_lag{p}')

# --- lag de clima (60 min atrás), para capturar tendência ---
df = deslocar(df, COLS_CLIMA, 4, '_lag4')

# --- variações (rampas): sinal de nebulosidade em movimento ---
df['D_IRRADIATION_1H']  = df['IRRADIATION'] - df['IRRADIATION_lag4']
df['D_AC_POWER_1H']     = df['AC_POWER'] - df['AC_POWER_lag4']
df['D_MODULE_TEMP_1H']  = df['MODULE_TEMPERATURE'] - df['MODULE_TEMPERATURE_lag4']
df['DELTA_TEMP']        = df['MODULE_TEMPERATURE'] - df['AMBIENT_TEMPERATURE']

# --- janelas móveis de 1h sobre AC_POWER (nível e volatilidade recentes) ---
df = df.sort_values(CHAVE + ['DATE_TIME'])
roll = df.set_index('DATE_TIME').groupby(CHAVE)['AC_POWER'].rolling('1h')
df['AC_MEDIA_1H'] = roll.mean().reset_index(level=[0, 1], drop=True).values
df['AC_DESVIO_1H'] = roll.std().reset_index(level=[0, 1], drop=True).fillna(0).values

# --- atributos temporais: minuto do dia em codificação cíclica ---
df['MINUTO_DIA'] = df['DATE_TIME'].dt.hour * 60 + df['DATE_TIME'].dt.minute
df['HORA']       = df['DATE_TIME'].dt.hour
df['SIN_DIA']    = np.sin(2 * np.pi * df['MINUTO_DIA'] / 1440)
df['COS_DIA']    = np.cos(2 * np.pi * df['MINUTO_DIA'] / 1440)
df['DIA_PERIODO'] = (df['DATE_TIME'] - df['DATE_TIME'].min()).dt.days

# --- remoção de bordas: qualquer NaN em coluna efetivamente usada torna a linha inválida ---
# Os lags/rampas são feitos por merge sobre DATE_TIME (correto na presença de lacunas), o que
# gera NaN sempre que o vizinho de 15/30/60 min não existe. Como o Ridge (E1) e a regressão
# logística (C1) não aceitam NaN nativamente, as linhas incompletas são removidas AQUI, uma vez,
# garantindo que as duas trilhas (A e B) treinem exatamente sobre a mesma base limpa.
COLS_MODELAGEM = [
    'AC_POWER', 'AC_POWER_lag1', 'AC_POWER_lag2', 'AC_POWER_lag4',
    'DC_POWER', 'DC_POWER_lag1', 'DC_POWER_lag2', 'DC_POWER_lag4',
    'IRRADIATION', 'IRRADIATION_lag4',
    'AMBIENT_TEMPERATURE', 'AMBIENT_TEMPERATURE_lag4',
    'MODULE_TEMPERATURE', 'MODULE_TEMPERATURE_lag4',
    'D_IRRADIATION_1H', 'D_AC_POWER_1H', 'D_MODULE_TEMP_1H', 'DELTA_TEMP',
    'AC_MEDIA_1H', 'AC_DESVIO_1H', 'SIN_DIA', 'COS_DIA', 'MINUTO_DIA',
    ALVO,
]

n_antes = len(df)
na_por_coluna = df[COLS_MODELAGEM].isna().sum()
na_por_coluna = na_por_coluna[na_por_coluna > 0].sort_values(ascending=False)

df = df.dropna(subset=COLS_MODELAGEM).reset_index(drop=True)

print(f"Linhas antes de remover bordas: {n_antes:,}".replace(',', '.'))
print(f"Linhas após remover bordas (lags/rampas/alvo indefinidos): {len(df):,}".replace(',', '.'))
print(f"Removidas: {n_antes - len(df):,} ({(n_antes - len(df)) / n_antes:.2%} da base)".replace(',', '.'))
if len(na_por_coluna):
    print("\nAusentes por coluna que motivaram a remoção (antes do dropna):")
    for col, q in na_por_coluna.items():
        print(f"   {col:28s}: {int(q):>6,}".replace(',', '.'))

# Verificação explícita: nenhuma coluna de modelagem deve conter NaN a partir daqui
assert df[COLS_MODELAGEM].isna().sum().sum() == 0, "Ainda há NaN em colunas de modelagem"
print("\nVerificação: 0 NaN nas colunas de modelagem. Base pronta para treino.")


Linhas antes de remover bordas: 136.476
Linhas após remover bordas (lags/rampas/alvo indefinidos): 133.253
Removidas: 3.223 (2.36% da base)

Ausentes por coluna que motivaram a remoção (antes do dropna):
   AC_POWER_lag4               :  1.405
   MODULE_TEMPERATURE_lag4     :  1.405
   AMBIENT_TEMPERATURE_lag4    :  1.405
   IRRADIATION_lag4            :  1.405
   DC_POWER_lag4               :  1.405
   D_IRRADIATION_1H            :  1.405
   D_AC_POWER_1H               :  1.405
   D_MODULE_TEMP_1H            :  1.405
   AC_POWER_H1                 :  1.405
   AC_POWER_lag2               :    837
   DC_POWER_lag2               :    837
   DC_POWER_lag1               :    496
   AC_POWER_lag1               :    496

Verificação: 0 NaN nas colunas de modelagem. Base pronta para treino.


C Split temporal e definição do evento operacional

O split é temporal, não aleatório: treino até 09/06/2020, teste de 10/06 a 17/06/2020. Um `train_test_split` aleatório colocaria vizinhos de 15 minutos em partições opostas, e a autocorrelação da série produziria uma métrica de teste otimista e sem sentido.

O evento `EVENTO_BAIXA` compara o alvo com a mediana histórica do mesmo horário e da mesma planta, calculada apenas no treino, nenhuma estatística do teste entra na definição do rótulo. Horários cuja mediana de treino é zero (noite) são excluídos da Trilha B: prever "não haverá geração às 2h da manhã" não é considerada uma tarefa.

In [18]:
treino_mask = df['DATE_TIME'] < DATA_CORTE
teste_mask  = ~treino_mask

# Normalização de DC_POWER pelo máximo de treino da própria planta (ver decisão (c))
dc_max_treino = df[treino_mask].groupby('PLANTA')['DC_POWER'].max()
for col in ['DC_POWER', 'DC_POWER_lag1', 'DC_POWER_lag2', 'DC_POWER_lag4']:
    df[f'{col}_NORM'] = df[col] / df['PLANTA'].map(dc_max_treino)

# Mediana de referência por (planta, minuto do dia), estimada SÓ no treino
ref = (df[treino_mask].groupby(['PLANTA', 'MINUTO_DIA'])[TARGET]
       .median().rename('MEDIANA_SLOT').reset_index())
df = df.merge(ref, on=['PLANTA', 'MINUTO_DIA'], how='left')
df['MEDIANA_SLOT'] = df['MEDIANA_SLOT'].fillna(0)

# Trilha B: rótulo binário e máscara de período diurno
df['DIURNO'] = df['MEDIANA_SLOT'] > 0
df['EVENTO_BAIXA'] = ((df[ALVO] < LIMIAR_BAIXA * df['MEDIANA_SLOT']) & df['DIURNO']).astype(int)

ATRIBUTOS = [
    'AC_POWER', 'AC_POWER_lag1', 'AC_POWER_lag2', 'AC_POWER_lag4',
    'DC_POWER_NORM', 'DC_POWER_lag1_NORM', 'DC_POWER_lag2_NORM', 'DC_POWER_lag4_NORM',
    'IRRADIATION', 'IRRADIATION_lag4',
    'AMBIENT_TEMPERATURE', 'AMBIENT_TEMPERATURE_lag4',
    'MODULE_TEMPERATURE', 'MODULE_TEMPERATURE_lag4',
    'D_IRRADIATION_1H', 'D_AC_POWER_1H', 'D_MODULE_TEMP_1H', 'DELTA_TEMP',
    'AC_MEDIA_1H', 'AC_DESVIO_1H',
    'SIN_DIA', 'COS_DIA', 'MINUTO_DIA', 'PLANTA', 'MEDIANA_SLOT',
]

# Ordenação cronológica global: pré-requisito do TimeSeriesSplit
treino = df[treino_mask].sort_values('DATE_TIME').reset_index(drop=True)
teste  = df[teste_mask].sort_values('DATE_TIME').reset_index(drop=True)

X_tr, y_tr = treino[ATRIBUTOS], treino[ALVO]
X_te, y_te = teste[ATRIBUTOS],  teste[ALVO]

# Trilha B: apenas linhas diurnas
tr_d, te_d = treino[treino['DIURNO']], teste[teste['DIURNO']]
Xc_tr, yc_tr = tr_d[ATRIBUTOS], tr_d['EVENTO_BAIXA']
Xc_te, yc_te = te_d[ATRIBUTOS], te_d['EVENTO_BAIXA']

print(f"Treino: {len(treino):,} linhas  ({treino['DATE_TIME'].min().date()} a {treino['DATE_TIME'].max().date()})".replace(',', '.'))
print(f"Teste : {len(teste):,} linhas  ({teste['DATE_TIME'].min().date()} a {teste['DATE_TIME'].max().date()})".replace(',', '.'))
print(f"Atributos: {len(ATRIBUTOS)}\n")
print("Trilha B - prevalência do evento de baixa geração (linhas diurnas):")
print(f"  treino: {yc_tr.mean():.2%}  ({yc_tr.sum():,} positivos de {len(yc_tr):,})".replace(',', '.'))
print(f"  teste : {yc_te.mean():.2%}  ({yc_te.sum():,} positivos de {len(yc_te):,})".replace(',', '.'))


Treino: 99.813 linhas  (2020-05-15 a 2020-06-09)
Teste : 33.440 linhas  (2020-06-10 a 2020-06-17)
Atributos: 25

Trilha B - prevalência do evento de baixa geração (linhas diurnas):
  treino: 30.92%  (16.627 positivos de 53.776)
  teste : 36.47%  (6.499 positivos de 17.820)


**1. Baseline**

O baseline define o piso, uma vez que o desempenho que qualquer modelo precisa superar este para justificar sua existência. Ou seja, não adianta ter um modelo complexo que não bate o baseline, logo ele será descartado.

**Trilha A (dois baselines):**

- Persistência (`ŷ(t+1h) = AC_POWER(t)`): o baseline canônico de séries temporais. É a hipótese "nada muda em uma hora". Difícil de bater em horizontes curtos e trivial de implementar — é o piso honesto.
- Climatologia por horário (`ŷ(t+1h) = mediana de treino do slot`): ignora o estado atual e usa apenas a curva média do dia. É o piso do "modelo que só sabe que horas são".

A comparação entre os dois já é informativa: se a persistência vence, a informação está no estado recente; se a climatologia vence, está no ciclo diurno.

**Trilha B (dois baselines):**
- Classe majoritária (`DummyClassifier`): nunca prevê evento. Acurácia alta, recall zero. Serve para expor porque acurácia não é uma métrica adaequada para este caso.
- Persistência do evento: prevê evento em `t+1h` se a geração em `t` já está abaixo do limiar. 

**Piso diurno vs. base completa.** `AC_POWER` é zero em cerca de metade das linhas (período noturno), e todo baseline acerta esses zeros trivialmente. O RMSE calculado sobre a base inteira é, portanto, diluído por essas linhas fáceis e superestima a qualidade do piso. Por isso o piso é reportado também restrito ao período diurno, sendo uma régua honesta, onde persistência e climatologia efetivamente competem e onde o ganho do modelo final precisa aparecer.

In [19]:
# --- Trilha A: baselines de regressão ---
base_persistencia_te  = teste['AC_POWER'].values
base_climatologia_te  = teste['MEDIANA_SLOT'].values
base_persistencia_tr  = treino['AC_POWER'].values
base_climatologia_tr  = treino['MEDIANA_SLOT'].values

linhas = []
for nome, p_tr, p_te in [('Persistência: ŷ(t+1h) = AC_POWER(t)', base_persistencia_tr, base_persistencia_te),
                         ('Climatologia: mediana do horário (treino)', base_climatologia_tr, base_climatologia_te)]:
    linhas.append({'baseline': nome,
                   'RMSE_treino': rmse(y_tr, p_tr), 'MAE_treino': mean_absolute_error(y_tr, p_tr),
                   'RMSE_teste': rmse(y_te, p_te),  'MAE_teste': mean_absolute_error(y_te, p_te),
                   'R2_teste': r2_score(y_te, p_te)})

baseline_reg = pd.DataFrame(linhas).round(3)
PISO_RMSE = baseline_reg['RMSE_teste'].min()
print(f"AC_POWER no teste: média = {y_te.mean():.1f} kW | desvio = {y_te.std():.1f} kW | máx = {y_te.max():.1f} kW\n")
display(baseline_reg)
print(f"\n>>> PISO (Trilha A - base completa): RMSE de teste a superar = {PISO_RMSE:.3f} kW")

# --- Decomposição do piso por período: diurno vs. noturno ---
# AC_POWER = 0 em ~metade das linhas (noite). Todo baseline acerta esses zeros trivialmente,
# então o RMSE da base completa é diluído pelas linhas fáceis. O piso relevante é o diurno.
mask_diurno_te = teste['DIURNO'].values
piso_por_periodo = []
for nome, p_te in [('Persistência', base_persistencia_te),
                   ('Climatologia', base_climatologia_te)]:
    for periodo, m in [('Diurno', mask_diurno_te), ('Noturno', ~mask_diurno_te), ('Completo', np.ones(len(y_te), bool))]:
        if m.sum() == 0:
            continue
        piso_por_periodo.append({
            'baseline': nome, 'período': periodo, 'n_linhas': int(m.sum()),
            'RMSE': rmse(y_te.values[m], p_te[m]),
            'MAE':  mean_absolute_error(y_te.values[m], p_te[m])})

piso_periodo = pd.DataFrame(piso_por_periodo)
PISO_RMSE_DIURNO = piso_periodo.query("período == 'Diurno'")['RMSE'].min()
pct_noturno = float((~mask_diurno_te).mean() * 100)

print(f"\nComposição do teste: {pct_noturno:.1f}% das linhas são noturnas (AC_POWER = 0, acerto trivial).")
display(piso_periodo.round(3).pivot_table(index='baseline', columns='período',
        values=['RMSE', 'MAE'], observed=True))
print(f">>> PISO relevante (Trilha A - período DIURNO): RMSE a superar = {PISO_RMSE_DIURNO:.3f} kW")
print("    O piso diurno é a régua honesta: é onde persistência e climatologia realmente competem,")
print("    e onde o modelo final precisa demonstrar ganho. Decomposição completa na Seção 4.")


AC_POWER no teste: média = 235.0 kW | desvio = 326.2 kW | máx = 1411.0 kW



,baseline,RMSE_treino,MAE_treino,RMSE_teste,MAE_teste,R2_teste
0,Persistência: ŷ(t+1h) = AC_POWER(t),220.124,124.619,189.643,108.218,0.662
1,Climatologia: mediana do horário (treino),242.345,141.863,233.127,137.183,0.489



>>> PISO (Trilha A - base completa): RMSE de teste a superar = 189.643 kW

Composição do teste: 46.7% das linhas são noturnas (AC_POWER = 0, acerto trivial).


MAE                      RMSE                 
período      Completo   Diurno Noturno Completo   Diurno Noturno
baseline                                                        
Climatologia  137.183  253.827   4.110  233.127  318.893  18.305
Persistência  108.218  199.457   4.128  189.643  259.221  18.308

>>> PISO relevante (Trilha A - período DIURNO): RMSE a superar = 259.221 kW
    O piso diurno é a régua honesta: é onde persistência e climatologia realmente competem,
    e onde o modelo final precisa demonstrar ganho. Decomposição completa na Seção 4.


In [20]:
# --- Trilha B: baselines de classificação ---
dummy = DummyClassifier(strategy='most_frequent').fit(Xc_tr, yc_tr)
pred_dummy = dummy.predict(Xc_te)

# persistência do evento: já está baixo agora -> estará baixo em 1h
pred_persist = ((te_d['AC_POWER'] < LIMIAR_BAIXA * te_d['MEDIANA_SLOT'])).astype(int).values

linhas = []
for nome, pred, score in [('Classe majoritária (nunca prevê evento)', pred_dummy, dummy.predict_proba(Xc_te)[:, 1]),
                          ('Persistência do evento', pred_persist, pred_persist.astype(float))]:
    linhas.append({'baseline': nome,
                   'acurácia': (pred == yc_te).mean(),
                   'F1_evento': f1_score(yc_te, pred, zero_division=0),
                   'ROC_AUC': roc_auc_score(yc_te, score)})

baseline_clf = pd.DataFrame(linhas).round(3)
PISO_AUC = baseline_clf['ROC_AUC'].max()
display(baseline_clf)
print(f"\n>>> PISO (Trilha B): ROC-AUC de teste a superar = {PISO_AUC:.3f}")
print("Observação: a classe majoritária atinge acurácia alta com F1 = 0 no evento de interesse.")
print("Acurácia é métrica inadequada sob desbalanceamento; ROC-AUC e F1 do evento são as métricas de decisão.")


,baseline,acurácia,F1_evento,ROC_AUC
0,Classe majoritária (nunca prevê evento),0.635,0.00,0.500
1,Persistência do evento,0.627,0.33,0.547



>>> PISO (Trilha B): ROC-AUC de teste a superar = 0.547
Observação: a classe majoritária atinge acurácia alta com F1 = 0 no evento de interesse.
Acurácia é métrica inadequada sob desbalanceamento; ROC-AUC e F1 do evento são as métricas de decisão.


**2. Experimentos**

Protocolo de validação: `TimeSeriesSplit(n_splits=5)` sobre o conjunto de treino ordenado cronologicamente. Cada fold treina no passado e valida no futuro imediato. `KFold` embaralhado é inválido aqui, pois validaria o modelo em instantes anteriores aos que ele viu no treino, medindo interpolação em vez de previsão.

Métrica primária: RMSE (Trilha A). Penaliza quadraticamente os erros grandes, que na operação da usina são exatamente os que importam: pois, um erro de 400 kW ao meio-dia "custa mais" que 40 erros de 10 kW ao amanhecer. Métrica secundária: MAE, interpretável na unidade física (kW médios de desvio).

Cada execução é reportada como **média ± desvio-padrão entre os 5 folds**. O desvio importa tanto quanto a média: um modelo com RMSE 40 ± 3 é operacionalmente preferível a um com RMSE 38 ± 15, porque o segundo tem semanas em que falha.

In [21]:
cv = TimeSeriesSplit(n_splits=N_SPLITS)
SCORING_REG = {'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error'}

experimentos_reg = {
    'E1 - Ridge (linear regularizado)': Pipeline([
        ('escala', StandardScaler()),
        ('modelo', Ridge(alpha=1.0, random_state=RANDOM_STATE))]),

    'E2 - Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=16, min_samples_leaf=5,
        n_jobs=-1, random_state=RANDOM_STATE),

    'E3 - HistGradientBoosting': HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.08, max_depth=None, max_leaf_nodes=31,
        l2_regularization=1.0, early_stopping=False, random_state=RANDOM_STATE),

    'E4 - HGB profundo (mais capacidade)': HistGradientBoostingRegressor(
        max_iter=600, learning_rate=0.05, max_leaf_nodes=63,
        min_samples_leaf=40, l2_regularization=2.0,
        early_stopping=False, random_state=RANDOM_STATE),
}

resultados = []
for nome, modelo in experimentos_reg.items():
    r = cross_validate(modelo, X_tr, y_tr, cv=cv, scoring=SCORING_REG, n_jobs=1)
    resultados.append({
        'experimento': nome,
        'RMSE_médio': -r['test_rmse'].mean(), 'RMSE_std': r['test_rmse'].std(),
        'MAE_médio':  -r['test_mae'].mean(),  'MAE_std':  r['test_mae'].std(),
        'tempo_fit_s': r['fit_time'].mean(),
    })
    print(f"{nome:42s}  RMSE = {-r['test_rmse'].mean():7.3f} ± {r['test_rmse'].std():5.3f}")

cv_reg = pd.DataFrame(resultados).sort_values('RMSE_médio').reset_index(drop=True)
cv_reg.round(3)


E1 - Ridge (linear regularizado)            RMSE = 181.592 ± 5.151
E2 - Random Forest                          RMSE = 171.334 ± 7.261
E3 - HistGradientBoosting                   RMSE = 172.326 ± 7.326
E4 - HGB profundo (mais capacidade)         RMSE = 174.433 ± 5.830


,experimento,RMSE_médio,RMSE_std,MAE_médio,MAE_std,tempo_fit_s
0,E2 - Random Forest,171.334,7.261,86.140,5.774,45.796
1,E3 - HistGradientBoosting,172.326,7.326,86.747,5.188,2.782
2,E4 - HGB profundo (mais capacidade),174.433,5.830,87.745,5.006,6.766
3,E1 - Ridge (linear regularizado),181.592,5.151,117.206,4.169,0.076


In [22]:
# E5 - busca de hiperparâmetros sobre a família vencedora (boosting), com CV temporal
espaco = {
    'max_iter':          [300, 500, 800],
    'learning_rate':     [0.03, 0.05, 0.08, 0.12],
    'max_leaf_nodes':    [31, 63, 127],
    'min_samples_leaf':  [20, 40, 80],
    'l2_regularization': [0.0, 1.0, 5.0],
}
busca = RandomizedSearchCV(
    HistGradientBoostingRegressor(early_stopping=False, random_state=RANDOM_STATE),
    param_distributions=espaco, n_iter=15, cv=cv,
    scoring='neg_root_mean_squared_error', random_state=RANDOM_STATE, n_jobs=-1, refit=True)
busca.fit(X_tr, y_tr)

i = busca.best_index_
rmse_e5, std_e5 = -busca.cv_results_['mean_test_score'][i], busca.cv_results_['std_test_score'][i]
print("E5 - melhores hiperparâmetros:")
for k, v in busca.best_params_.items():
    print(f"   {k:20s} = {v}")
print(f"\nE5 - RMSE CV = {rmse_e5:.3f} ± {std_e5:.3f}")

r_e5 = cross_validate(busca.best_estimator_, X_tr, y_tr, cv=cv, scoring=SCORING_REG, n_jobs=1)
cv_reg = pd.concat([cv_reg, pd.DataFrame([{
    'experimento': 'E5 - HGB ajustado (RandomizedSearchCV)',
    'RMSE_médio': -r_e5['test_rmse'].mean(), 'RMSE_std': r_e5['test_rmse'].std(),
    'MAE_médio':  -r_e5['test_mae'].mean(),  'MAE_std':  r_e5['test_mae'].std(),
    'tempo_fit_s': r_e5['fit_time'].mean()}])], ignore_index=True).sort_values('RMSE_médio').reset_index(drop=True)

cv_reg['RMSE (média ± std)'] = cv_reg.apply(lambda l: f"{l['RMSE_médio']:.3f} ± {l['RMSE_std']:.3f}", axis=1)
cv_reg['MAE (média ± std)']  = cv_reg.apply(lambda l: f"{l['MAE_médio']:.3f} ± {l['MAE_std']:.3f}", axis=1)
cv_reg[['experimento', 'RMSE (média ± std)', 'MAE (média ± std)', 'tempo_fit_s']].round(3)


E5 - melhores hiperparâmetros:
   min_samples_leaf     = 20
   max_leaf_nodes       = 31
   max_iter             = 300
   learning_rate        = 0.03
   l2_regularization    = 1.0

E5 - RMSE CV = 168.851 ± 7.429


,experimento,RMSE (média ± std),MAE (média ± std),tempo_fit_s
0,E5 - HGB ajustado (RandomizedSearchCV),168.851 ± 7.429,85.422 ± 5.518,2.452
1,E2 - Random Forest,171.334 ± 7.261,86.140 ± 5.774,45.796
2,E3 - HistGradientBoosting,172.326 ± 7.326,86.747 ± 5.188,2.782
3,E4 - HGB profundo (mais capacidade),174.433 ± 5.830,87.745 ± 5.006,6.766
4,E1 - Ridge (linear regularizado),181.592 ± 5.151,117.206 ± 4.169,0.076


In [23]:
# --- Trilha B: experimentos de classificação, mesmo CV temporal ---
experimentos_clf = {
    'C1 - Regressão logística': Pipeline([
        ('escala', StandardScaler()),
        ('modelo', LogisticRegression(max_iter=2000, class_weight='balanced',
                                      random_state=RANDOM_STATE))]),
    'C2 - Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=16, min_samples_leaf=5, class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_STATE),
    'C3 - HistGradientBoosting': HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.08, max_leaf_nodes=31, l2_regularization=1.0,
        early_stopping=False, random_state=RANDOM_STATE),
}

res_clf = []
for nome, modelo in experimentos_clf.items():
    r = cross_validate(modelo, Xc_tr, yc_tr, cv=cv,
                       scoring={'auc': 'roc_auc', 'f1': 'f1'}, n_jobs=1)
    res_clf.append({'experimento': nome,
                    'ROC_AUC (média ± std)': f"{r['test_auc'].mean():.3f} ± {r['test_auc'].std():.3f}",
                    'F1_evento (média ± std)': f"{r['test_f1'].mean():.3f} ± {r['test_f1'].std():.3f}",
                    'auc_medio': r['test_auc'].mean()})
    print(f"{nome:30s}  AUC = {r['test_auc'].mean():.3f} ± {r['test_auc'].std():.3f}")

cv_clf = pd.DataFrame(res_clf).sort_values('auc_medio', ascending=False).reset_index(drop=True)
cv_clf.drop(columns='auc_medio')


C1 - Regressão logística        AUC = 0.932 ± 0.017
C2 - Random Forest              AUC = 0.939 ± 0.013
C3 - HistGradientBoosting       AUC = 0.930 ± 0.015


,experimento,ROC_AUC (média ± std),F1_evento (média ± std)
0,C2 - Random Forest,0.939 ± 0.013,0.807 ± 0.030
1,C1 - Regressão logística,0.932 ± 0.017,0.792 ± 0.035
2,C3 - HistGradientBoosting,0.930 ± 0.015,0.801 ± 0.033
